# XE9L Algorithms at 0x8D99

Analyze the 8 algorithms loaded by `load_algo_8d99`, `load_algo_8e19`, `load_algo_8e99`, `load_algo_8f19`.

These are **8 × 32-instruction algorithms** at 44kHz.

| Function | ROM Address | Algorithms | A-RAM Slots |
|----------|-------------|------------|-------------|
| load_algo_8d99 | 0x8D99, 0x8DD9 | 2 | 0x00, 0x20 |
| load_algo_8e19 | 0x8E19, 0x8E59 | 2 | 0x40, 0x60 |
| load_algo_8e99 | 0x8E99, 0x8ED9 | 2 | 0x80, 0xA0 |
| load_algo_8f19 | 0x8F19, 0x8F59 | 2 | 0xC0, 0xE0 |

**Note**: The first pair (0x8D99 + 0x8DD9) may use a register-passing trick where slot 0x00 leaves values in A/B registers for slot 0x20 to use.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from sam8905_interpreter import (
    SAM8905Interpreter,
    plot_waveform,
    export_wav,
    print_state,
)
from sam8905_aram_decoder import decode_algorithm, analyze_dram_usage

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)

# Load XE9L ROM
ROM_PATH = "/home/jeff/bastel/roms/hohner/ms4/xe9l_v141.bin"
with open(ROM_PATH, 'rb') as f:
    rom = f.read()

print(f"ROM size: {len(rom)} bytes")

ROM size: 65536 bytes


## Extract All 8 Algorithms

Each algorithm is 32 words (64 bytes) at 44kHz.

In [2]:
def extract_algorithm(rom, addr):
    """Extract 32 x 15-bit words from ROM (little-endian byte pairs)."""
    words = []
    for i in range(32):
        offset = addr + (i * 2)
        low = rom[offset]
        high = rom[offset + 1]
        word = low | (high << 8)
        words.append(word)
    return words

# 8 algorithms, 32 words each, 64 bytes apart
ALGO_BASE = 0x8D99
ALGO_SIZE = 64  # bytes
SLOT_NAMES = ['0x00', '0x20', '0x40', '0x60', '0x80', '0xA0', '0xC0', '0xE0']

algorithms = []
for i in range(8):
    addr = ALGO_BASE + (i * ALGO_SIZE)
    algorithms.append({
        'index': i,
        'addr': addr,
        'slot': SLOT_NAMES[i],
        'words': extract_algorithm(rom, addr)
    })

# Summary
print("Algorithms from 0x8D99 (8 × 32 instructions, 44kHz):")
print("=" * 65)
for algo in algorithms:
    nop_count = sum(1 for w in algo['words'] if w == 0x7FFF)
    active = 32 - nop_count
    print(f"  {algo['index']}: 0x{algo['addr']:04X} -> slot {algo['slot']}  ({active:2d} active, {nop_count:2d} NOP)")

Algorithms from 0x8D99 (8 × 32 instructions, 44kHz):
  0: 0x8D99 -> slot 0x00  (18 active, 14 NOP)
  1: 0x8DD9 -> slot 0x20  ( 8 active, 24 NOP)
  2: 0x8E19 -> slot 0x40  ( 5 active, 27 NOP)
  3: 0x8E59 -> slot 0x60  ( 0 active, 32 NOP)
  4: 0x8E99 -> slot 0x80  ( 9 active, 23 NOP)
  5: 0x8ED9 -> slot 0xA0  ( 0 active, 32 NOP)
  6: 0x8F19 -> slot 0xC0  (17 active, 15 NOP)
  7: 0x8F59 -> slot 0xE0  ( 0 active, 32 NOP)


## Raw Algorithm Data

In [3]:
for algo in algorithms:
    nop_count = sum(1 for w in algo['words'] if w == 0x7FFF)
    print(f"\nAlgorithm {algo['index']} at 0x{algo['addr']:04X} (slot {algo['slot']}) - {32-nop_count} active:")
    for row in range(0, 32, 8):
        chunk = algo['words'][row:row+8]
        print(f"  {row:2d}: " + " ".join(f"{w:04X}" for w in chunk))


Algorithm 0 at 0x8D99 (slot 0x00) - 18 active:
   0: 7BEF 7FFF 087F 28BF 7AEF 30FD 006F 7FFF
   8: 7FFF 7FFF 7FFF 7FFF 7FFF 10BF 02DF 19E7
  16: 20BF 7C7F 22DF 087F 28BF 7AEF 30FD 006F
  24: 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 10BF

Algorithm 1 at 0x8DD9 (slot 0x20) - 8 active:
   0: 02DF 19E7 093F 0ADB 1BDF 207F 7CBF 22DF
   8: 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF
  16: 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF
  24: 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF

Algorithm 2 at 0x8E19 (slot 0x40) - 5 active:
   0: 016F 08BF 11F7 02DF 7FFF 7FFF 7EFE 7FFF
   8: 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF
  16: 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF
  24: 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF

Algorithm 3 at 0x8E59 (slot 0x60) - 0 active:
   0: 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF
   8: 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF
  16: 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF
  24: 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF 7FFF

Algorithm 4 at 0x8E99 (slot 0x80) - 9 active:
   0: 00EF 207F 7EFB 087D 10

## Decode All Algorithms

In [4]:
for algo in algorithms:
    nop_count = sum(1 for w in algo['words'] if w == 0x7FFF)
    if nop_count == 32:
        print(f"\n{'=' * 60}")
        print(f"Algorithm {algo['index']} at 0x{algo['addr']:04X} (slot {algo['slot']})")
        print("  [ALL NOP - empty algorithm]")
        continue
    
    print(f"\n{'=' * 60}")
    print(f"Algorithm {algo['index']} at 0x{algo['addr']:04X} (slot {algo['slot']})")
    print("=" * 60)
    print(decode_algorithm(algo['words']))
    
    print("\nD-RAM Usage:")
    usage = analyze_dram_usage(algo['words'])
    for addr, counts in sorted(usage.items()):
        print(f"  D[{addr:2d}]: read={counts['read']}, write={counts['write']}")


Algorithm 0 at 0x8D99 (slot 0x00)

PC00: 7BEF  RADD, <WPHI, WSP> ***
PC01: 7FFF  RSP, <WSP> ***
PC02: 087F  RM 1, <WA>
PC03: 28BF  RM 5, <WB>
PC04: 7AEF  RADD, <WPHI>
PC05: 30FD  RM 6, <WWF>
PC06: 006F  RM 0, <WA, WPHI>
PC07: 7FFF  RSP, <WSP> ***
PC08: 7FFF  RSP, <WSP> ***
PC09: 7FFF  RSP, <WSP> ***
PC10: 7FFF  RSP, <WSP> ***
PC11: 7FFF  RSP, <WSP> ***
PC12: 7FFF  RSP, <WSP> ***
PC13: 10BF  RM 2, <WB>
PC14: 02DF  RADD 0, <WM>
PC15: 19E7  RM 3, <WPHI, WXY, WSP> ***
PC16: 20BF  RM 4, <WB>
PC17: 7C7F  RP, <WA>
PC18: 22DF  RADD 4, <WM>
PC19: 087F  RM 1, <WA>
PC20: 28BF  RM 5, <WB>
PC21: 7AEF  RADD, <WPHI>
PC22: 30FD  RM 6, <WWF>
PC23: 006F  RM 0, <WA, WPHI>
PC24: 7FFF  RSP, <WSP> ***
PC25: 7FFF  RSP, <WSP> ***
PC26: 7FFF  RSP, <WSP> ***
PC27: 7FFF  RSP, <WSP> ***
PC28: 7FFF  RSP, <WSP> ***
PC29: 7FFF  RSP, <WSP> ***
PC30: 7FFF  RSP, <WSP> ***
PC31: 10BF  RM 2, <WB>

D-RAM Usage:
  D[ 0]: read=2, write=1
  D[ 1]: read=2, write=0
  D[ 2]: read=2, write=0
  D[ 3]: read=1, write=0
  D[ 4]: re

## Analyze Register-Passing Pair (Slots 0x00 + 0x20)

The first two algorithms may work as a cooperating pair:
- Slot 0x00 (0x8D99) computes intermediate values and leaves them in A/B registers
- Slot 0x20 (0x8DD9) picks up those values and continues processing

Look for patterns:
- Algo 0 ending without writing A/B to D-RAM
- Algo 1 starting without loading A/B from D-RAM

In [5]:
algo0 = algorithms[0]['words']
algo1 = algorithms[1]['words']

print("Algorithm 0 (0x8D99) - last 8 instructions:")
for pc in range(24, 32):
    w = algo0[pc]
    if w == 0x7FFF:
        print(f"  PC{pc:02d}: {w:04X}  NOP")
    elif w & 0x4000:
        addr = (w >> 8) & 0x0F
        print(f"  PC{pc:02d}: {w:04X}  RADD {addr} (writes to D-RAM)")
    else:
        addr = (w >> 8) & 0x0F
        ops = []
        if w & 0x0080: ops.append('WA')
        if w & 0x0040: ops.append('WB')
        print(f"  PC{pc:02d}: {w:04X}  RM {addr}, ops={ops}")

print("\nAlgorithm 1 (0x8DD9) - first 8 instructions:")
for pc in range(8):
    w = algo1[pc]
    if w == 0x7FFF:
        print(f"  PC{pc:02d}: {w:04X}  NOP")
    elif w & 0x4000:
        addr = (w >> 8) & 0x0F
        print(f"  PC{pc:02d}: {w:04X}  RADD {addr} (uses accumulator)")
    else:
        addr = (w >> 8) & 0x0F
        ops = []
        if w & 0x0080: ops.append('WA')
        if w & 0x0040: ops.append('WB')
        if not (w & 0x00C0): ops.append('(no WA/WB - uses existing A/B)')
        print(f"  PC{pc:02d}: {w:04X}  RM {addr}, ops={ops}")

Algorithm 0 (0x8D99) - last 8 instructions:
  PC24: 7FFF  NOP
  PC25: 7FFF  NOP
  PC26: 7FFF  NOP
  PC27: 7FFF  NOP
  PC28: 7FFF  NOP
  PC29: 7FFF  NOP
  PC30: 7FFF  NOP
  PC31: 10BF  RM 0, ops=['WA']

Algorithm 1 (0x8DD9) - first 8 instructions:
  PC00: 02DF  RM 2, ops=['WA', 'WB']
  PC01: 19E7  RM 9, ops=['WA', 'WB']
  PC02: 093F  RM 9, ops=['(no WA/WB - uses existing A/B)']
  PC03: 0ADB  RM 10, ops=['WA', 'WB']
  PC04: 1BDF  RM 11, ops=['WA', 'WB']
  PC05: 207F  RM 0, ops=['WB']
  PC06: 7CBF  RADD 12 (uses accumulator)
  PC07: 22DF  RM 2, ops=['WA', 'WB']


## Operation Summary

In [6]:
def count_operations(words):
    wacc = sum(1 for w in words if (w & 0x0008) and not (w & 0x4000))
    wwf = sum(1 for w in words if (w & 0x0001) and not (w & 0x4000))
    wphi = sum(1 for w in words if (w & 0x0020) and not (w & 0x4000))
    wsp = sum(1 for w in words if (w & 0x0004) and not (w & 0x4000))
    radd = sum(1 for w in words if (w & 0x4000))
    nop = sum(1 for w in words if w == 0x7FFF)
    return {'Active': 32-nop, 'NOP': nop, 'WACC': wacc, 'WWF': wwf, 'WPHI': wphi, 'WSP': wsp, 'RADD': radd}

print("Operation Summary:")
print("=" * 80)
print(f"{'Idx':<4} {'Addr':<8} {'Slot':<6} {'Active':<7} {'NOP':<5} {'WACC':<5} {'WWF':<5} {'WPHI':<5} {'RADD':<5}")
print("-" * 80)

for algo in algorithms:
    ops = count_operations(algo['words'])
    print(f"{algo['index']:<4} 0x{algo['addr']:04X}  {algo['slot']:<6} {ops['Active']:<7} {ops['NOP']:<5} {ops['WACC']:<5} {ops['WWF']:<5} {ops['WPHI']:<5} {ops['RADD']:<5}")

Operation Summary:
Idx  Addr     Slot   Active  NOP   WACC  WWF   WPHI  RADD 
--------------------------------------------------------------------------------
0    0x8D99  0x00   18      14    13    14    12    18   
1    0x8DD9  0x20   8       24    6     7     3     25   
2    0x8E19  0x40   5       27    3     4     3     28   
3    0x8E59  0x60   0       32    0     0     0     32   
4    0x8E99  0x80   9       23    6     7     6     25   
5    0x8ED9  0xA0   0       32    0     0     0     32   
6    0x8F19  0xC0   17      15    10    12    9     20   
7    0x8F59  0xE0   0       32    0     0     0     32   


## Notes

### Algorithm Structure

These are synthesis algorithms loaded as alternatives to `load_algo_table_1`.

**Register-passing trick (slots 0x00 + 0x20)**:
- When two consecutive sound slots use cooperating algorithms, the first can leave
  intermediate values in the A and B registers
- The second slot picks up those values without needing to load from D-RAM
- This effectively doubles processing power for complex synthesis

### Empty Algorithms

Slots filled with NOP (0x7FFF) are disabled - no sound output from those slots.